In [ ]:

# full event level
# fp = f"./365day_future_prediction_outputs_50_full_stage_filter_v8"

# subset event level
n = 100
# fp = f"./365day_future_prediction_outputs_50_subset_{n}_stage_filter_v8"
#------
# full patient level 
fp = "./365day_future_prediction_outputs_50_full_stage_filter_patient_level_v2"

# subset patient level
# fp = "./365day_future_prediction_outputs_50_subset_1000_stage_filter_patient_level_v2"



In [ ]:
dirs = [
    "/LSTM_365DayFutureTarget_detailed_outputs.csv",
    "/MLP_365DayFutureTarget_detailed_outputs.csv",
    "/RNN_365DayFutureTarget_detailed_outputs.csv",
    "/TCN_365DayFutureTarget_detailed_outputs.csv",
    "/Transformer_365DayFutureTarget_detailed_outputs.csv",
]

# dirs = [
#     "/DeepSurv_LSTM_365DayFutureTarget_detailed_outputs.csv",
#     "/DeepSurv_MLP_365DayFutureTarget_detailed_outputs.csv",
#     "/DeepSurv_RNN_365DayFutureTarget_detailed_outputs.csv",
#     "/DeepSurv_TCN_365DayFutureTarget_detailed_outputs.csv",
#     "/DeepSurv_Transformer_365DayFutureTarget_detailed_outputs.csv",
# ]

In [ ]:
# no eskd survival model

# baseline/eskd

# eskd, full event level
# fp_eskd = f"./365day_future_prediction_outputs_full_stage_filter_eskd_v2"

# eskd, subset event level
n2 = 100
# fp_eskd = f"./365day_future_prediction_outputs_subset_{n2}_stage_filter_eskd_v2"

# dirs_eskd = ['/XGBoost_365DayFuture_Classifier_detailed_outputs_classification.csv']

# ---------
# eskd, full patient level
fp_eskd = f"./365day_future_prediction_outputs_full_stage_filter_eskd_v2_patient_level"
dirs_eskd = ['/XGBoost_365DayFuture_Classifier_detailed_outputs_classification_pt_lvl.csv']

['/XGBoost_365DayFuture_Classifier_detailed_outputs_classification.csv',
#  '/XGBoost_TTE_Survival_detailed_outputs_survival.csv',
#  'xgboost_only_365day_future_switch_analysis.csv'
 ]

In [ ]:
filepaths2 = [fp_eskd + i for i in dirs_eskd]
print(filepaths2)

filepaths1 = [fp + i for i in dirs]
print(filepaths1)

# modifier dependent <<<<<
# no eskd survival model
filepaths = filepaths1 + filepaths2
# filepaths = filepaths1 
# filepaths = filepaths2
print(filepaths)

In [ ]:
import logging
import sys
import os

# modifier = "deepsurv" if "DeepSurv" in dirs[0] else "classification"
log_path = f'results_logs/{fp}_results_{modifier}_metrics_v3_output.log'

out_folder = "results_files_v3"
out_path =  os.path.join(out_folder, fp.strip("./"))
print(out_path)
try:
    os.mkdir(out_path)
except FileExistsError:
    pass

In [ ]:
results_path = os.path.join(out_path, f"{modifier}_results.csv")
print(results_path)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams
from matplotlib.ticker import FormatStrFormatter

# ---------------------------------------------------------------------
# Load custom font
# ---------------------------------------------------------------------
font_path = "Google.ttf"
font_manager.fontManager.addfont(font_path)
custom_font = font_manager.FontProperties(fname=font_path).get_name()

rcParams["font.family"] = custom_font
rcParams["axes.unicode_minus"] = False


# GREENS = ["#E8F5E9", "#A5D6A7", "#66BB6A", "#2E7D32"]
# HATCHES = ["", "/", "//", "///"]

# BLUES = ["#E3F2FD", "#BBDEFB", "#90CAF9", "#42A5F5", "#1565C0"]
shades = ["#2E86AB", "#A23B72", "#F18F01", "#C73E1D", "#6A994E"]
# shades = ["#8E44AD", "#16A085", "#E67E22", "#C0392B", "#2980B9"]
HATCHES = ["", "/", "//", "xx", "oo"]


plt.rcParams.update({
    "axes.linewidth": 1.0,
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    'lines.linewidth': 1
})



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import argparse
from concurrent.futures import ProcessPoolExecutor
import torch

from sklearn.metrics import (
    precision_recall_curve, roc_curve,
    average_precision_score, roc_auc_score,
    f1_score, precision_score, recall_score,
    confusion_matrix
)
from scipy.special import softmax
from sklearn.utils import resample

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

def load_and_process_file(filepath):
    df = pd.read_csv(filepath)
    logits = df[["cl_logit_0", "cl_logit_1"]].values
    probs = softmax(logits, axis=1)[:, 1]
    labels = df["cl_true_label"].astype(int).values
    return probs, labels

def find_optimal_threshold(y_true, y_probs, max_threshold=1.0, step=0.01):
    thresholds = np.arange(0.0, max_threshold + step, step)
    best_f1 = -1.0
    best_thresh = 0.0
    for t in thresholds:
        preds = (y_probs >= t).astype(int)
        if np.sum(preds) == 0:
            continue
        f1 = f1_score(y_true, preds)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t
    return best_thresh

def compute_confusion_matrix(y_true, y_preds):
    tn, fp, fn, tp = confusion_matrix(y_true, y_preds).ravel()
    confusion_m = {
        'True Negative': tn,
        'False Positive': fp,
        'False Negative': fn,
        'True Positive': tp
    }
    return confusion_m

def compute_metrics(y_true, y_probs, threshold):
    preds = (y_probs >= threshold).astype(int)
    metrics = {
        "AUROC": roc_auc_score(y_true, y_probs),
        "AUPRC": average_precision_score(y_true, y_probs),
        "F1": f1_score(y_true, preds),
        "PPV": precision_score(y_true, preds),
        "Recall": recall_score(y_true, preds),
        "Avg Precision": average_precision_score(y_true, y_probs),
        "Avg Recall": recall_score(y_true, preds),
        "Confusion Matrix": compute_confusion_matrix(y_true, preds)
    }
    return metrics

def bootstrap_metrics_gpu(y_true, y_probs, threshold, n_iterations=1000, gpu_id=3):

    device = torch.device(f"cuda:{gpu_id}" if torch.cuda.is_available() else "cpu")
    print(f"Bootstrapping on {torch.cuda.get_device_name(device)} (ID: {gpu_id})")

    y_true_gpu = torch.tensor(y_true, dtype=torch.float32, device=device)
    y_probs_gpu = torch.tensor(y_probs, dtype=torch.float32, device=device)
    n_samples = len(y_true)
    
    boot_results = {k: [] for k in ["AUROC", "AUPRC", "F1", "PPV", "Recall", "Avg Precision", "Avg Recall"]}
    
    # Process in batches of 100 iterations to maximize parallel math
    batch_size = 1024
    for b in range(0, n_iterations, batch_size):
        current_batch_size = min(batch_size, n_iterations - b)
        
        # 2. Vectorized Resampling: Generate (Batch x 1.45M) indices
        indices = torch.randint(0, n_samples, (current_batch_size, n_samples), device=device)
        
        # 3. Massively Parallel Metric Calculation
        # These operations happen across all 100 bootstrap samples simultaneously
        yb_true = y_true_gpu[indices]
        yb_probs = y_probs_gpu[indices]
        
        preds = (yb_probs >= threshold).float()
        tp = (preds * yb_true).sum(dim=1)
        fp = (preds * (1 - yb_true)).sum(dim=1)
        fn = ((1 - preds) * yb_true).sum(dim=1)
        
        precision = tp / (tp + fp + 1e-7)
        recall = tp / (tp + fn + 1e-7)
        f1 = 2 * (precision * recall) / (precision + recall + 1e-7)
        
        boot_results["PPV"].extend(precision.cpu().tolist())
        boot_results["Recall"].extend(recall.cpu().tolist())
        boot_results["Avg Recall"].append(recall.cpu().tolist())
        boot_results["F1"].extend(f1.cpu().tolist())

        # 4. Complex Metrics (AUROC/AUPRC) 
        # Sklearn is still the gold standard for these; we compute them per sample
        for i in range(current_batch_size):
            y_t_cpu = yb_true[i].cpu().numpy()
            y_p_cpu = yb_probs[i].cpu().numpy()
            auroc = roc_auc_score(y_t_cpu, y_p_cpu)
            auprc = average_precision_score(y_t_cpu, y_p_cpu)
            boot_results["AUROC"].append(auroc)
            boot_results["AUPRC"].append(auprc)
            boot_results["Avg Precision"].append(auprc)

    final_stats = {}
    for k, v in boot_results.items():

        mean = np.mean(v)
        low = np.percentile(v, 2.5)
        high = np.percentile(v, 97.5)
        
        # Store raw numbers for plotting/math
        final_stats[k] = (mean, low, high)

    return final_stats

def bootstrap_metrics_gpu_0(y_true, y_probs, threshold, n_iterations=1000, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print(f"Bootstrapping with {n_iterations} iterations on {device}...")
    
    # Move data to GPU
    y_true_gpu = torch.tensor(y_true, dtype=torch.float32, device=device)
    y_probs_gpu = torch.tensor(y_probs, dtype=torch.float32, device=device)
    n_samples = len(y_true)
    
    # Initialize containers for results
    boot_results = {k: [] for k in ["AUROC", "AUPRC", "F1", "PPV", "Recall", "Avg Precision", "Avg Recall"]}

    for i in range(n_iterations):
        # 1. Resample indices on GPU
        indices = torch.randint(0, n_samples, (n_samples,), device=device)
        yb_true = y_true_gpu[indices]
        yb_probs = y_probs_gpu[indices]
        
        # 2. Binary predictions
        preds = (yb_probs >= threshold).float()
        
        # 3. Calculate basic metrics (F1, PPV, Recall) on GPU
        tp = (preds * yb_true).sum()
        fp = (preds * (1 - yb_true)).sum()
        fn = ((1 - preds) * yb_true).sum()
        
        precision = tp / (tp + fp + 1e-7)
        recall = tp / (tp + fn + 1e-7)
        f1 = 2 * (precision * recall) / (precision + recall + 1e-7)
        
        boot_results["PPV"].append(precision.item())
        boot_results["Recall"].append(recall.item())
        boot_results["Avg Recall"].append(recall.item())
        boot_results["F1"].append(f1.item())

        # 4. Calculate AUROC/AUPRC 
        # (Note: Standard sklearn requires CPU; we move only the small resampled batch back)
        yb_true_cpu = yb_true.cpu().numpy()
        yb_probs_cpu = yb_probs.cpu().numpy()
        
        auroc = roc_auc_score(yb_true_cpu, yb_probs_cpu)
        auprc = average_precision_score(yb_true_cpu, yb_probs_cpu)
        
        boot_results["AUROC"].append(auroc)
        boot_results["AUPRC"].append(auprc)
        boot_results["Avg Precision"].append(auprc)

    # 5. Compute mean and 95% Confidence Intervals
    final_metrics = {}
    for k, v in boot_results.items():
        mean = np.mean(v)
        low = np.percentile(v, 2.5)
        high = np.percentile(v, 97.5)
        final_metrics[k] = (mean, low, high)
        
    print("Bootstrapping completed.")
    return final_metrics

def bootstrap_once(seed, y_true, y_probs, threshold):
    np.random.seed(seed)
    idx = resample(np.arange(len(y_true)))
    yb_true, yb_probs = y_true[idx], y_probs[idx]
    preds = (yb_probs >= threshold).astype(int)
    return {
        "AUROC": roc_auc_score(yb_true, yb_probs),
        "AUPRC": average_precision_score(yb_true, yb_probs),
        "F1": f1_score(yb_true, preds),
        "PPV": precision_score(yb_true, preds),
        "Recall": recall_score(yb_true, preds),
        "Avg Precision": average_precision_score(yb_true, yb_probs),
        "Avg Recall": recall_score(yb_true, preds),
        "Confusion Matrix": confusion_matrix(y_true, preds) # different format for boostrapping
    }

def bootstrap_metrics(y_true, y_probs, threshold, n_iterations=1000, n_workers=8):
    print(f"Bootstrapping with {n_iterations} iterations using {n_workers} workers...")
    boot_metrics = {k: [] for k in ["AUROC", "AUPRC", "F1", "PPV", "Recall", "Avg Precision", "Avg Recall",
                                        "Confusion Matrix"]}
    seeds = np.random.randint(0, 100000, size=n_iterations)

    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        futures = [executor.submit(bootstrap_once, s, y_true, y_probs, threshold) for s in seeds]
        for f in futures:
            result = f.result()
            for k, v in result.items():
                boot_metrics[k].append(v)

    print("Bootstrapping completed.")
    return {k: (np.mean(v), np.percentile(v, 2.5), np.percentile(v, 97.5)) for k, v in boot_metrics.items()}

def plot_roc_pr_curves_v0(results):
    # Plot ROC curves on the same figure
    plt.figure()
    for name, res in results.items():
        fpr, tpr, _ = roc_curve(res['y_true'], res['y_probs'])
        auroc = res['metrics']['AUROC']
        plt.plot(fpr, tpr, label=f"{name} (AUROC={auroc:.3f})")
    plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")

    title = "ROC Curves"
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    figname = f"{modifier}_{title}.png"
    figpath = os.path.join(out_path, figname)
    plt.savefig(figpath)
    plt.show()

    
    # Plot PR curves on the same figure
    plt.figure()
    for name, res in results.items():
        precision, recall, _ = precision_recall_curve(res['y_true'], res['y_probs'])
        auprc = res['metrics']['AUPRC']
        plt.plot(recall, precision, label=f"{name} (AUPRC={auprc:.3f})")
    plt.xlabel("Recall")
    plt.ylabel("Precision")

    title = "Precision-Recall Curves"
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    figname = f"{modifier}_{title}.png"
    figpath = os.path.join(out_path, figname)
    plt.savefig(figpath)
    plt.show()

def plot_roc_pr_curves_l2(results):
    # Plot ROC curves on the same figure
    ####
    l2 = "legend_right"
    fig, ax = plt.subplots(figsize=(14, 6))
    # fig, ax = plt.subplots()
    for i, (name, res) in enumerate(results.items()):
        fpr, tpr, _ = roc_curve(res['y_true'], res['y_probs'])
        auroc = res['metrics']['AUROC']
        color = shades[i] if i < len(shades) else None
        ax.plot(fpr, tpr, color=color, label=f"{name} (AUROC={auroc:.3f})")
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, linewidth=1)
    ax.set_xlabel("False Positive Rate", fontsize=13, fontweight="semibold")
    ax.set_ylabel("True Positive Rate", fontsize=13, fontweight="semibold")
    title = "ROC Curves"
    ax.set_title(title, fontsize=12, fontweight="semibold")
    ####
    ax.legend(prop=font_manager.FontProperties(fname=font_path, size=11)
        ,frameon=True, edgecolor='#1a1a1a', fancybox=False
        ,loc='center left', bbox_to_anchor=(1.02, 0.5))
    ax.tick_params(axis="both", labelsize=10)
    ax.grid(axis="both", linestyle="--", linewidth=0.6, alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    figname = f"{modifier}_{title}_{l2}.png"
    figpath = os.path.join(out_path, figname)
    plt.savefig(figpath, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Plot PR curves on the same figure
    ###
    fig, ax = plt.subplots(figsize=(14, 6))
    # fig, ax = plt.subplots()
    for i, (name, res) in enumerate(results.items()):
        precision, recall, _ = precision_recall_curve(res['y_true'], res['y_probs'])
        auprc = res['metrics']['AUPRC']
        color = shades[i] if i < len(shades) else None
        ax.plot(recall, precision, color=color, label=f"{name} (AUPRC={auprc:.3f})")
    ax.set_xlabel("Recall", fontsize=13, fontweight="semibold")
    ax.set_ylabel("Precision", fontsize=13, fontweight="semibold")

    title = "Precision-Recall Curves"
    ax.set_title(title, fontsize=12, fontweight="semibold")
    ####
    ax.legend(prop=font_manager.FontProperties(fname=font_path, size=11)
        ,frameon=True, edgecolor='#1a1a1a', fancybox=False
        ,loc='center left', bbox_to_anchor=(1.02, 0.5))
    ax.tick_params(axis="both", labelsize=10)
    ax.grid(axis="both", linestyle="--", linewidth=0.6, alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    figname = f"{modifier}_{title}_{l2}.png"
    figpath = os.path.join(out_path, figname)
    plt.savefig(figpath, dpi=300, bbox_inches='tight')
    plt.show()


def plot_roc_pr_curves(results):
    # Plot ROC curves on the same figure
    ####
    # fig, ax = plt.subplots(figsize=(14, 6))
    fig, ax = plt.subplots()
    for i, (name, res) in enumerate(results.items()):
        fpr, tpr, _ = roc_curve(res['y_true'], res['y_probs'])
        auroc = res['metrics']['AUROC']
        color = shades[i] if i < len(shades) else None
        ax.plot(fpr, tpr, color=color, label=f"{name} (AUROC={auroc:.3f})")
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, linewidth=1)
    ax.set_xlabel("False Positive Rate", fontsize=13, fontweight="semibold")
    ax.set_ylabel("True Positive Rate", fontsize=13, fontweight="semibold")
    title = "ROC Curves"
    ax.set_title(title, fontsize=12, fontweight="semibold")
    ax.legend(prop=font_manager.FontProperties(fname=font_path, size=11),frameon=True, edgecolor='#1a1a1a', fancybox=False)
    ####
    # ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5))
    ax.tick_params(axis="both", labelsize=10)
    ax.grid(axis="both", linestyle="--", linewidth=0.6, alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    figname = f"{modifier}_{title}.png"
    figpath = os.path.join(out_path, figname)
    plt.savefig(figpath, dpi=300, bbox_inches='tight')
    plt.show()
    
    # Plot PR curves on the same figure
    ###
    # fig, ax = plt.subplots(figsize=(14, 6))
    fig, ax = plt.subplots()
    for i, (name, res) in enumerate(results.items()):
        precision, recall, _ = precision_recall_curve(res['y_true'], res['y_probs'])
        auprc = res['metrics']['AUPRC']
        color = shades[i] if i < len(shades) else None
        ax.plot(recall, precision, color=color, label=f"{name} (AUPRC={auprc:.3f})")
    ax.set_xlabel("Recall", fontsize=13, fontweight="semibold")
    ax.set_ylabel("Precision", fontsize=13, fontweight="semibold")

    title = "Precision-Recall Curves"
    ax.set_title(title, fontsize=12, fontweight="semibold")
    ax.legend(prop=font_manager.FontProperties(fname=font_path, size=11),frameon=True, edgecolor='#1a1a1a', fancybox=False)
    #####
    # ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5))
    ax.tick_params(axis="both", labelsize=10)
    ax.grid(axis="both", linestyle="--", linewidth=0.6, alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    figname = f"{modifier}_{title}.png"
    figpath = os.path.join(out_path, figname)
    plt.savefig(figpath, dpi=300, bbox_inches='tight')
    plt.show()


def plot_metric_bars(all_boot_metrics):
    metrics = ["F1", "PPV", "Recall", "Avg Precision", "Avg Recall"]
    model_names = list(all_boot_metrics.keys())

    for metric in metrics:
        means = [all_boot_metrics[m][metric][0] for m in model_names]
        lowers = [all_boot_metrics[m][metric][0] - all_boot_metrics[m][metric][1] for m in model_names]
        uppers = [all_boot_metrics[m][metric][2] - all_boot_metrics[m][metric][0] for m in model_names]

        plt.figure()
        plt.bar(model_names, means, yerr=[lowers, uppers], capsize=5)
        plt.ylabel(metric)

        title = f"{metric} Comparison with 95% CI"
        plt.title(title) # fix
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        figname = f"{modifier}_{title}.png"
        figpath = os.path.join(out_path, figname)
        plt.savefig(figpath)
        plt.show()


def evaluate_models(filepaths, modifier, n_boot=1000, n_workers=8, threshold_cap=1.0, threshold_step=0.01, verbose=False):
    results = {}
    for filepath in filepaths:
        name = os.path.splitext(os.path.basename(filepath))[0]

        if modifier == 'deepsurv':
            name = modifier.capitalize() + '_' + name.split("_")[1]
        else:
            name = modifier.capitalize() + '_' + name.split("_")[0]

        print(f"\nProcessing file: {name}")
        y_probs, y_true = load_and_process_file(filepath)
        threshold = find_optimal_threshold(y_true, y_probs, max_threshold=threshold_cap, step=threshold_step)
        
        print(f"Optimal threshold selected: {threshold:.3f}")
        metrics = compute_metrics(y_true, y_probs, threshold)

        print("Metrics: ")
        for keys,values in metrics.items():
            print(keys, values)

        results[name] = {
            "y_true": y_true,
            "y_probs": y_probs,
            "threshold": threshold,
            "metrics": metrics
        }


    plot_roc_pr_curves(results)
    plot_roc_pr_curves_l2(results)
    

    return results

In [ ]:
from contextlib import redirect_stdout, redirect_stderr

with open(log_path, 'w') as f:
    with redirect_stdout(f), redirect_stderr(f):
        metrics = evaluate_models(filepaths, modifier, threshold_cap = 0.0239, n_workers=20, verbose = True)
        # metrics = evaluate_models_polars( filepaths, threshold_cap = 0.0239, n_workers=20, verbose = True)

# log_file.close()

In [ ]:
metrics